# EGFR WT vs. T790M – Gefitinib & Osimertinib
**Frage:** Warum behandelt dasselbe Medikament WT-EGFR und die T790M-Mutante unterschiedlich?

Pipeline (aus den Vorlesungen 3–6): PDB → Pocket/Box (wie `measure center` in ChimeraX) → **Redocking-Validierung** (RMSD < 2 Å) → Docking beider Liganden in WT und T790M → Interaktionsanalyse (Met793-H-Brücke, Abstand zu Rest 790, Warhead–Cys797).

Ausführen: *Laufzeit → Alle ausführen*. Ergebnisse landen in `results.json` und `docking_results.csv` – diese Zahlen kommen in die Folien.

In [ ]:
!pip -q install rdkit vina meeko biopython pandas matplotlib requests
import os, json, subprocess, requests, numpy as np, pandas as pd, matplotlib.pyplot as plt
from Bio.PDB import PDBParser, PDBIO, Select, NeighborSearch
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate
from vina import Vina
os.makedirs('work', exist_ok=True)

## 1  Konfiguration
`ligand_code` = 3-Buchstaben-Code des Co-Kristall-Liganden (IRE = Gefitinib, YY3 = Osimertinib). **Bitte vor dem Lauf mit RCSB prüfen** (Folie 2 der Projektpräsentation nennt für Osimertinib fälschlich „YYE“; 6JXT ist laut RCSB WT + AZD9291/YY3, nicht Gefitinib).

In [ ]:
LIGANDS = {
  'Gefitinib':   'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',
  'Osimertinib': 'COc1cc(N(C)CCN(C)C)c(NC(=O)C=C)cc1Nc1nccc(-c2cn(C)c3ccccc23)n1',
}
# Rezeptor-Set: (PDB, Zustand, Ko-Ligand-Code, Kette)
RECEPTORS = [
  ('4WKQ','WT',   'IRE','A'),   # WT + Gefitinib, 1.85 A
  ('6JXT','WT',   'YY3','A'),   # WT + Osimertinib, 2.31 A
  ('6JX4','T790M','YY3','A'),   # T790M + Osimertinib (soaking)
  ('3UG2','G719S/T790M','IRE','A'),  # Doppelmutante + Gefitinib (kein reiner T790M!)
]
BOX = 22.0          # Boxkante in Angstrom
EXHAUSTIVENESS = 16
SEED = 42

## 2  Strukturen laden und vorbereiten

In [ ]:
def fetch(pdb):
    p=f'work/{pdb}.pdb'
    if not os.path.exists(p):
        r=requests.get(f'https://files.rcsb.org/download/{pdb}.pdb',timeout=60); r.raise_for_status()
        open(p,'w').write(r.text)
    return p

class ProtSel(Select):
    def __init__(s,chain): s.chain=chain
    def accept_chain(s,c): return c.id==s.chain
    def accept_residue(s,r): return r.id[0]==' '          # nur Protein, keine Wasser/HETATM
    def accept_atom(s,a): return a.get_altloc() in (' ','A')

class LigSel(Select):
    def __init__(s,chain,code): s.chain,s.code=chain,code
    def accept_chain(s,c): return c.id==s.chain
    def accept_residue(s,r): return r.get_resname()==s.code
    def accept_atom(s,a): return a.get_altloc() in (' ','A')

def prepare(pdb,code,chain):
    path=fetch(pdb); st=PDBParser(QUIET=True).get_structure(pdb,path)
    io=PDBIO(); io.set_structure(st)
    io.save(f'work/{pdb}_prot.pdb',ProtSel(chain)); io.save(f'work/{pdb}_lig.pdb',LigSel(chain,code))
    lig=Chem.MolFromPDBFile(f'work/{pdb}_lig.pdb',removeHs=False)
    assert lig is not None and lig.GetNumAtoms()>0, f'Ligand {code} in {pdb} nicht gefunden'
    centre=lig.GetConformer().GetPositions().mean(axis=0)   # = ChimeraX: measure center :CODE
    return centre

info={}
for pdb,state,code,ch in RECEPTORS:
    info[pdb]=dict(state=state,code=code,centre=prepare(pdb,code,ch).tolist())
    print(pdb,state,code,'Box-Zentrum',np.round(info[pdb]['centre'],2))

In [ ]:
# Wasserstoffe + Rezeptor-PDBQT (Meeko). Falls mk_prepare_receptor.py fehlt: pip install -U meeko
!pip -q install pdbfixer openmm 2>/dev/null
from pdbfixer import PDBFixer
from openmm.app import PDBFile
for pdb,state,code,ch in RECEPTORS:
    fx=PDBFixer(f'work/{pdb}_prot.pdb'); fx.findMissingResidues(); fx.missingResidues={}
    fx.findNonstandardResidues(); fx.replaceNonstandardResidues(); fx.findMissingAtoms(); fx.addMissingAtoms()
    fx.addMissingHydrogens(7.4)
    PDBFile.writeFile(fx.topology,fx.positions,open(f'work/{pdb}_protH.pdb','w'),keepIds=True)
    c=info[pdb]['centre']
    cmd=['mk_prepare_receptor.py','--read_pdb',f'work/{pdb}_protH.pdb','-o',f'work/{pdb}_rec','-p',
         '--box_center',*map(str,np.round(c,3)),'--box_size',str(BOX),str(BOX),str(BOX),'--allow_bad_res']
    r=subprocess.run(cmd,capture_output=True,text=True); print(pdb,'OK' if r.returncode==0 else r.stderr[-400:])

## 3  Liganden vorbereiten (Protonierungszustand bewusst wählen!)

In [ ]:
def lig_pdbqt(smiles,name,protonate=None):
    m=Chem.AddHs(Chem.MolFromSmiles(smiles)); AllChem.EmbedMolecule(m,randomSeed=SEED); AllChem.MMFFOptimizeMolecule(m)
    prep=MoleculePreparation(); setup=prep.prepare(m)[0]
    s,ok,err=PDBQTWriterLegacy.write_string(setup); assert ok,err
    open(f'work/{name}.pdbqt','w').write(s); return m
for n,s in LIGANDS.items(): lig_pdbqt(s,n)

## 4  Docking-Funktion und Interaktionsanalyse

In [ ]:
def dock(rec_pdb,lig_name,centre,ex=EXHAUSTIVENESS,n_poses=9,lig_file=None):
    v=Vina(sf_name='vina',seed=SEED,verbosity=0)
    v.set_receptor(f'work/{rec_pdb}_rec.pdbqt'); v.set_ligand_from_file(lig_file or f'work/{lig_name}.pdbqt')
    v.compute_vina_maps(center=list(centre),box_size=[BOX]*3)
    v.dock(exhaustiveness=ex,n_poses=n_poses)
    out=f'work/{rec_pdb}_{lig_name}_out.pdbqt'; v.write_poses(out,n_poses=n_poses,overwrite=True)
    return v.energies(n_poses=n_poses)[:,0], out

def poses(pdbqt):
    pm=PDBQTMolecule.from_file(pdbqt,skip_typing=True)
    mol=RDKitMolCreate.from_pdbqt_mol(pm)[0]; return mol

def struct(pdb): return PDBParser(QUIET=True).get_structure(pdb,f'work/{pdb}_protH.pdb')[0]

def residue(model,resi):
    for r in model.get_residues():
        if r.id[1]==resi and r.id[0]==' ': return r

def interactions(pdb,mol,conf=0):
    """Met793-H-Brücke, Kontakte zu Rest 790, Warhead-C...Cys797-SG"""
    model=struct(pdb); xyz=mol.GetConformer(conf).GetPositions()
    sym=[a.GetSymbol() for a in mol.GetAtoms()]
    res={}
    met=residue(model,793); N=met['N'].coord
    acc=[i for i,s in enumerate(sym) if s=='N']
    res['d_Met793N_ligN']=float(min(np.linalg.norm(xyz[i]-N) for i in acc))      # <= 3.2 A -> H-Brücke zum Hinge
    r790=residue(model,790); res['res790']=r790.get_resname()
    heavy790=[a.coord for a in r790 if a.element!='H']
    d=np.linalg.norm(np.array(heavy790)[:,None,:]-xyz[None,:,:],axis=2)
    res['d_res790_min']=float(d.min()); res['n_contacts_res790_4A']=int((d.min(axis=0)<4.0).sum())
    patt=Chem.MolFromSmarts('[CH2]=[CH]C(=O)N')
    if mol.HasSubstructMatch(patt):
        beta=mol.GetSubstructMatch(patt)[0]; sg=residue(model,797)['SG'].coord
        res['d_warhead_Cys797SG']=float(np.linalg.norm(xyz[beta]-sg))             # < 4 A = reaktionsfähig
    return res

def rmsd_to_crystal(pdb,code,lig_name,pose_mol):
    ref=Chem.MolFromPDBFile(f'work/{pdb}_lig.pdb',removeHs=True)
    tmpl=Chem.RemoveHs(Chem.MolFromSmiles(LIGANDS[lig_name]))
    ref=AllChem.AssignBondOrdersFromTemplate(tmpl,ref)
    prb=Chem.RemoveHs(pose_mol)
    return rdMolAlign.CalcRMS(prb,ref)      # symmetrie-korrigiert, ohne Neu-Ausrichtung

## 5  Redocking (Validierung) – wenn RMSD > 2 Å, ist die Methode für die Frage ungeeignet

In [ ]:
redock=[]
for pdb,state,code,ch in RECEPTORS:
    lig_name='Gefitinib' if code=='IRE' else 'Osimertinib'
    sc,out=dock(pdb,lig_name,info[pdb]['centre']); mol=poses(out)
    # alle Posen einzeln gegen Kristall prüfen:
    rms=[]
    for cid in range(mol.GetNumConformers()):
        m1=Chem.Mol(mol); c=m1.GetConformer(cid); m1.RemoveAllConformers(); m1.AddConformer(c,assignId=True)
        rms.append(rmsd_to_crystal(pdb,code,lig_name,m1))
    redock.append(dict(pdb=pdb,state=state,ligand=lig_name,top1_rmsd=rms[0],best_rmsd=min(rms),
                       top1_score=float(sc[0]),success_top1=rms[0]<=2.0,success_top9=min(rms)<=2.0))
redock=pd.DataFrame(redock); print(redock.round(2).to_string(index=False)); redock.to_csv('redocking.csv',index=False)
if not redock.success_top1.any(): print('\n>>> ACHTUNG: Redocking scheitert überall – Docking-Scores NICHT interpretieren (Failure-Folie!)')

## 6  Docking beider Liganden in alle Rezeptoren

In [ ]:
rows=[]
for pdb,state,code,ch in RECEPTORS:
    for lig_name in LIGANDS:
        sc,out=dock(pdb,lig_name,info[pdb]['centre']); mol=poses(out)
        m1=Chem.Mol(mol); c=m1.GetConformer(0); m1.RemoveAllConformers(); m1.AddConformer(c,assignId=True)
        r=dict(pdb=pdb,state=state,ligand=lig_name,vina_top1=float(sc[0]),vina_top3_mean=float(np.mean(sc[:3])))
        r.update(interactions(pdb,m1,0)); rows.append(r)
res=pd.DataFrame(rows); res.to_csv('docking_results.csv',index=False); print(res.round(2).to_string(index=False))

In [ ]:
# Plot: Score je Rezeptor und Ligand
piv=res.pivot(index='pdb',columns='ligand',values='vina_top1').loc[[r[0] for r in RECEPTORS]]
ax=piv.plot.bar(figsize=(7,4),color=['#2E6F95','#D1495B']); ax.set_ylabel('Vina-Score (kcal/mol, negativer = besser)')
ax.set_title('Gleiche Methode, vier Strukturen – Score nicht mit Affinität gleichsetzen'); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig('docking_scores.png',dpi=200); plt.show()
json.dump(dict(redocking=redock.to_dict('records'),docking=res.to_dict('records')),open('results.json','w'),indent=1)

## 7  ChimeraX-Befehle für die Folienbilder (Vorlesung 4)
```
open 4wkq
name ligand :IRE            # bei 6JXT / 6JX4: :YY3
measure center :IRE         # = Box-Zentrum (nicht das SwissDock-Zentrum benutzen!)
show cartoon; color bychain
select ~ligand & ligand :<5 ; color magenta sel      # Tasche im 5-Å-Radius
hbonds ligand restrict any reveal true               # H-Brücken Ligand-Protein
surface; transparency 70
distance /A:793@N <Ligand-N>                          # Met793-Distanz
distance /A:797@SG <Warhead-C>                        # nur Osimertinib
mmaker #2 to #1                                       # WT/T790M überlagern, Rest 790 vergleichen
save pocket.png transparentBackground true
```
## 8  Sidequests (optional)
- SwissADME / ADMETlab: BOILED-Egg, CYP-Inhibition, hERG für beide Liganden.
- Osimertinib-Protonierungszustand (basisches Amin, pH 7.4) vs. neutral docken – ändert sich das Ranking?
- Kovalentes Docking (Vina kann nur nicht-kovalent): Cys797 als Flex-Rest oder mit AutoDock-CovDock/GNINA-Alternative.